In [2]:
!pip install -q transformers datasets accelerate

In [3]:


from __future__ import annotations

import argparse
import ast
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


DEFAULT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DEFAULT_SYSTEM_PROMPT = "You are an expert Python competitive programmer."


@dataclass
class Task:
    task_id: str
    prompt: str
    test: str
    entry_point: str


@dataclass
class TokenTrace:
    position: int
    token_id: int
    token_text: str
    entropy: float
    margin: float
    top1_id: int
    top1_text: str
    top2_id: int
    top2_text: str


@dataclass
class Generation:
    token_ids: list[int]
    code: str
    trace: list[TokenTrace]
    latency_s: float


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model-name", default=DEFAULT_MODEL)
    parser.add_argument("--output-dir", default="/kaggle/working/top2_counterfactual_pilot")
    parser.add_argument("--task-ids", default="", help="Comma-separated HumanEval task IDs.")
    parser.add_argument("--num-tasks", type=int, default=10, help="Used only when --task-ids is empty.")
    parser.add_argument("--max-new-tokens", type=int, default=256)
    parser.add_argument("--candidates-per-task", type=int, default=3)
    parser.add_argument("--controls-per-task", type=int, default=3)
    parser.add_argument("--edge-buffer", type=int, default=5)
    parser.add_argument("--timeout-s", type=int, default=8)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--trust-remote-code", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    # Jupyter/Kaggle executes a cell with an internal ``-f <kernel.json>``
    # argument. Accept that one argument pair while keeping normal CLI typos
    # visible to the user.
    args, unknown = parser.parse_known_args()
    if unknown:
        if len(unknown) == 2 and unknown[0] == "-f":
            return args
        parser.error(f"unrecognized arguments: {' '.join(unknown)}")
    return args


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_tasks(task_ids: str, num_tasks: int) -> list[Task]:
    dataset = load_dataset("openai_humaneval", split="test")
    requested_ids = {item.strip() for item in task_ids.split(",") if item.strip()}
    tasks = [
        Task(
            task_id=row["task_id"],
            prompt=row["prompt"],
            test=row["test"],
            entry_point=row["entry_point"],
        )
        for row in dataset
        if not requested_ids or row["task_id"] in requested_ids
    ]
    if requested_ids:
        missing = requested_ids - {task.task_id for task in tasks}
        if missing:
            raise ValueError(f"Unknown HumanEval task IDs: {sorted(missing)}")
        return tasks
    return tasks[:num_tasks]


def build_prompt(task: Task, tokenizer: Any) -> str:
    user_prompt = (
        "Complete the following Python function.\n"
        "Return only valid Python code. Do not use Markdown. Do not explain.\n\n"
        f"{task.prompt}"
    )
    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"{DEFAULT_SYSTEM_PROMPT}\n\n{user_prompt}"


def strip_markdown_fences(text: str) -> str:
    text = (text or "").strip()
    blocks = re.findall(r"```(?:python|py)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    blocks = [block.strip() for block in blocks if block.strip()]
    if blocks:
        keywords = ("def ", "import ", "from ", "class ", "return ", "assert ")
        return max(blocks, key=lambda block: sum(key in block for key in keywords) * 10 + len(block))
    return text.replace("```python", "").replace("```py", "").replace("```", "").strip()


def extract_code(raw_output: str, entry_point: str) -> str:
    text = strip_markdown_fences(raw_output)
    for marker in ("Explanation:", "Example:", "Examples:", "# Explanation"):
        marker_index = text.find(marker)
        if marker_index != -1:
            text = text[:marker_index].strip()
    match = re.search(rf"def\s+{re.escape(entry_point)}\s*\(", text)
    if match:
        imports = [
            line.strip()
            for line in text[: match.start()].splitlines()
            if line.strip().startswith(("import ", "from "))
        ]
        function_code = text[match.start() :].strip()
        return "\n".join(imports + ([""] if imports else []) + [function_code]).strip()
    return text.strip()


def evaluate(task: Task, raw_output: str, timeout_s: int) -> tuple[bool, str | None]:
    code = extract_code(raw_output, task.entry_point)
    try:
        ast.parse(code)
    except SyntaxError as error:
        return False, f"SyntaxError: {error.msg} at line {error.lineno}"

    prelude = (
        "from typing import *\nimport math\nimport re\nimport itertools\n"
        "import collections\nimport functools\nimport heapq\nimport bisect\n"
        "import string\nimport statistics\nfrom collections import *\n\n"
    )
    source = prelude + code + "\n\n" + task.test + f"\n\ncheck({task.entry_point})\n"
    with tempfile.TemporaryDirectory() as temp_dir:
        candidate_path = Path(temp_dir) / "candidate.py"
        candidate_path.write_text(source, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, str(candidate_path)],
                cwd=temp_dir,
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return False, f"Timeout: exceeded {timeout_s}s"
    if result.returncode == 0:
        return True, None
    stderr = (result.stderr or result.stdout or "unknown execution failure").strip()
    return False, stderr[-800:]


def model_input_device(model: Any) -> torch.device:
    return model.get_input_embeddings().weight.device


def entropy_and_top2(logits: torch.Tensor, tokenizer: Any, position: int) -> TokenTrace:
    logits = logits.float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    entropy = float((-(probabilities * log_probabilities).sum()).item())
    top_values, top_ids = torch.topk(logits, k=2, dim=-1)
    top1_id, top2_id = int(top_ids[0].item()), int(top_ids[1].item())
    return TokenTrace(
        position=position,
        token_id=top1_id,
        token_text=tokenizer.decode([top1_id]),
        entropy=entropy,
        margin=float((top_values[0] - top_values[1]).item()),
        top1_id=top1_id,
        top1_text=tokenizer.decode([top1_id]),
        top2_id=top2_id,
        top2_text=tokenizer.decode([top2_id]),
    )


@torch.inference_mode()
def greedy_generate(model: Any, tokenizer: Any, prompt: str, max_new_tokens: int) -> Generation:
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    started = time.perf_counter()
    outputs = model(prompt_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    generated_ids: list[int] = []
    trace: list[TokenTrace] = []
    eos_token_id = tokenizer.eos_token_id

    for position in range(max_new_tokens):
        token_trace = entropy_and_top2(next_logits[0], tokenizer, position)
        trace.append(token_trace)
        next_token_id = token_trace.top1_id
        generated_ids.append(next_token_id)
        if eos_token_id is not None and next_token_id == eos_token_id:
            break
        next_token = torch.tensor([[next_token_id]], device=input_device, dtype=torch.long)
        outputs = model(next_token, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    return Generation(
        token_ids=generated_ids,
        code=tokenizer.decode(generated_ids, skip_special_tokens=True),
        trace=trace,
        latency_s=time.perf_counter() - started,
    )


@torch.inference_mode()
def generate_top1_top2_batch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    top1_id: int,
    top2_id: int,
    max_new_tokens: int,
) -> tuple[Generation, Generation]:
    """Continue top-1 and top-2 branches in a batch of two equally long prefixes."""

    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=input_device).unsqueeze(0)
    shared_prefix = torch.cat([prompt_ids, prefix_tensor], dim=1) if prefix_ids else prompt_ids
    candidates = torch.tensor([[top1_id], [top2_id]], dtype=torch.long, device=input_device)
    input_ids = torch.cat([shared_prefix.repeat(2, 1), candidates], dim=1)
    started = time.perf_counter()
    outputs = model(input_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    branch_ids = [[top1_id], [top2_id]]
    branch_trace: list[list[TokenTrace]] = [[], []]
    finished = [False, False]
    eos_token_id = tokenizer.eos_token_id
    remaining = max(max_new_tokens - len(prefix_ids) - 1, 0)

    for step in range(remaining):
        next_ids: list[int] = []
        for branch_index in range(2):
            token_trace = entropy_and_top2(next_logits[branch_index], tokenizer, len(prefix_ids) + 1 + step)
            branch_trace[branch_index].append(token_trace)
            token_id = eos_token_id if finished[branch_index] and eos_token_id is not None else token_trace.top1_id
            branch_ids[branch_index].append(int(token_id))
            if eos_token_id is not None and token_id == eos_token_id:
                finished[branch_index] = True
            next_ids.append(int(token_id))
        if all(finished):
            break
        next_tensor = torch.tensor(next_ids, dtype=torch.long, device=input_device).unsqueeze(1)
        outputs = model(next_tensor, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    latency_s = time.perf_counter() - started
    complete_ids = [prefix_ids + branch for branch in branch_ids]
    return (
        Generation(
            token_ids=complete_ids[0],
            code=tokenizer.decode(complete_ids[0], skip_special_tokens=True),
            trace=branch_trace[0],
            latency_s=latency_s,
        ),
        Generation(
            token_ids=complete_ids[1],
            code=tokenizer.decode(complete_ids[1], skip_special_tokens=True),
            trace=branch_trace[1],
            latency_s=latency_s,
        ),
    )


def select_positions(
    trace: list[TokenTrace],
    candidates_per_task: int,
    controls_per_task: int,
    edge_buffer: int,
    rng: np.random.Generator,
) -> list[tuple[str, TokenTrace]]:
    valid = trace[edge_buffer : max(len(trace) - edge_buffer, edge_buffer)]
    high_entropy = sorted(valid, key=lambda item: item.entropy, reverse=True)[:candidates_per_task]
    high_positions = {item.position for item in high_entropy}
    controls_pool = [item for item in valid if item.position not in high_positions]
    controls_count = min(controls_per_task, len(controls_pool))
    controls = (
        [controls_pool[index] for index in rng.choice(len(controls_pool), size=controls_count, replace=False)]
        if controls_count
        else []
    )
    return [("high_entropy", item) for item in high_entropy] + [("random_control", item) for item in controls]


def baseline_record(task: Task, generation: Generation, passed: bool, error: str | None) -> dict[str, Any]:
    return {
        "task_id": task.task_id,
        "passed": passed,
        "error": error,
        "code": generation.code,
        "token_ids": generation.token_ids,
        "trace": [asdict(item) for item in generation.trace],
        "generation_latency_s": generation.latency_s,
    }


def write_summary(branch_records: list[dict[str, Any]], output_dir: Path) -> None:
    rows = []
    for selection_type in ("high_entropy", "random_control"):
        group = [record for record in branch_records if record["selection_type"] == selection_type]
        if not group:
            continue
        recoverable = sum(bool(record["recoverable"]) for record in group)
        rows.append(
            {
                "selection_type": selection_type,
                "n_positions": len(group),
                "recoverable": recoverable,
                "recovery_rate": recoverable / len(group),
                "mean_entropy": float(np.mean([record["entropy"] for record in group])),
                "mean_extra_tokens": float(np.mean([record["extra_tokens"] for record in group])),
            }
        )
    report = {"rows": rows, "created_at_unix": time.time()}
    (output_dir / "summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    markdown = ["# Top-1 vs Top-2 counterfactual pilot", "", "| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |", "|---|---:|---:|---:|---:|---:|"]
    markdown.extend(
        "| {selection_type} | {n_positions} | {recoverable} | {recovery_rate:.1%} | {mean_entropy:.3f} | {mean_extra_tokens:.1f} |".format(**row)
        for row in rows
    )
    (output_dir / "summary.md").write_text("\n".join(markdown) + "\n", encoding="utf-8")
    print("\n".join(markdown))


def main() -> None:
    args = parse_args()
def run_notebook(
    *,
    task_ids: str,
    model_name: str = DEFAULT_MODEL,
    output_dir: str = "/kaggle/working/top2_counterfactual_pilot",
    max_new_tokens: int = 256,
    candidates_per_task: int = 3,
    controls_per_task: int = 3,
    edge_buffer: int = 5,
    timeout_s: int = 8,
    seed: int = 42,
    trust_remote_code: bool = False,
    overwrite: bool = False,
) -> None:
    """Run the pilot directly from a Kaggle notebook cell.

    Example:
        run_notebook(task_ids="HumanEval/26,HumanEval/38", candidates_per_task=2)
    """

    args = argparse.Namespace(
        model_name=model_name,
        output_dir=output_dir,
        task_ids=task_ids,
        num_tasks=10,
        max_new_tokens=max_new_tokens,
        candidates_per_task=candidates_per_task,
        controls_per_task=controls_per_task,
        edge_buffer=edge_buffer,
        timeout_s=timeout_s,
        seed=seed,
        trust_remote_code=trust_remote_code,
        overwrite=overwrite,
    )
    main(args)


def main(args: argparse.Namespace | None = None) -> None:
    args = args or parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("This pilot requires a Kaggle GPU session.")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    rng = np.random.default_rng(args.seed)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    baseline_path = output_dir / "baselines.jsonl"
    branch_path = output_dir / "branches.jsonl"
    metadata_path = output_dir / "metadata.json"
    if args.overwrite:
        for path in (baseline_path, branch_path, metadata_path):
            if path.exists():
                path.unlink()

    tasks = load_tasks(args.task_ids, args.num_tasks)
    metadata_path.write_text(
        json.dumps({"args": vars(args), "tasks": [task.task_id for task in tasks]}, indent=2),
        encoding="utf-8",
    )
    print(f"Loading one model across {torch.cuda.device_count()} GPU(s): {args.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=args.trust_remote_code)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=args.trust_remote_code,
    )
    model.eval()

    existing_baselines = {record["task_id"]: record for record in read_jsonl(baseline_path)}
    existing_branches = {record["task_id"] for record in read_jsonl(branch_path)}
    for task in tasks:
        if task.task_id not in existing_baselines:
            print(f"Baseline {task.task_id}")
            prompt = build_prompt(task, tokenizer)
            generation = greedy_generate(model, tokenizer, prompt, args.max_new_tokens)
            passed, error = evaluate(task, generation.code, args.timeout_s)
            record = baseline_record(task, generation, passed, error)
            append_jsonl(baseline_path, record)
            existing_baselines[task.task_id] = record
            print(f"  {'PASS' if passed else 'FAIL'} | {len(generation.token_ids)} tokens")

        baseline = existing_baselines[task.task_id]
        if baseline["passed"]:
            print(f"Skip {task.task_id}: baseline passed (pilot targets failures).")
            continue
        if task.task_id in existing_branches:
            print(f"Skip {task.task_id}: branches already saved.")
            continue

        prompt = build_prompt(task, tokenizer)
        trace = [TokenTrace(**item) for item in baseline["trace"]]
        selected = select_positions(
            trace,
            candidates_per_task=args.candidates_per_task,
            controls_per_task=args.controls_per_task,
            edge_buffer=args.edge_buffer,
            rng=rng,
        )
        if not selected:
            print(f"Skip {task.task_id}: too few generated tokens for valid branch positions.")
            continue
        print(f"Branching {task.task_id}: {len(selected)} positions")
        for selection_type, point in selected:
            prefix_ids = baseline["token_ids"][: point.position]
            top1_generation, top2_generation = generate_top1_top2_batch(
                model,
                tokenizer,
                prompt,
                prefix_ids=prefix_ids,
                top1_id=point.top1_id,
                top2_id=point.top2_id,
                max_new_tokens=args.max_new_tokens,
            )
            top1_passed, top1_error = evaluate(task, top1_generation.code, args.timeout_s)
            top2_passed, top2_error = evaluate(task, top2_generation.code, args.timeout_s)
            record = {
                "task_id": task.task_id,
                "selection_type": selection_type,
                "position": point.position,
                "position_relative": point.position / max(len(baseline["token_ids"]) - 1, 1),
                "entropy": point.entropy,
                "margin": point.margin,
                "top1_token": point.top1_text,
                "top2_token": point.top2_text,
                "top1_passed": top1_passed,
                "top1_error": top1_error,
                "top2_passed": top2_passed,
                "top2_error": top2_error,
                "recoverable": (not top1_passed) and top2_passed,
                "top1_matches_baseline": top1_generation.code == baseline["code"],
                "extra_tokens": len(top1_generation.token_ids) + len(top2_generation.token_ids) - 2 * len(prefix_ids),
                "branch_latency_s": top1_generation.latency_s,
                "top1_code": top1_generation.code,
                "top2_code": top2_generation.code,
            }
            append_jsonl(branch_path, record)
            print(
                f"  {selection_type} t={point.position:>3} H={point.entropy:.3f} "
                f"top1={'P' if top1_passed else 'F'} top2={'P' if top2_passed else 'F'}"
            )
        existing_branches.add(task.task_id)
        write_summary(read_jsonl(branch_path), output_dir)

    write_summary(read_jsonl(branch_path), output_dir)
    print(f"\nSaved resumable outputs to: {output_dir}")


if __name__ == "__main__":
    main()

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading one model across 2 GPU(s): Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Skip HumanEval/0: baseline passed (pilot targets failures).
Skip HumanEval/1: baseline passed (pilot targets failures).
Skip HumanEval/2: baseline passed (pilot targets failures).
Skip HumanEval/3: baseline passed (pilot targets failures).
Skip HumanEval/4: baseline passed (pilot targets failures).
Skip HumanEval/5: baseline passed (pilot targets failures).
Skip HumanEval/6: baseline passed (pilot targets failures).
Skip HumanEval/7: baseline passed (pilot targets failures).
Skip HumanEval/8: baseline passed (pilot targets failures).
Skip HumanEval/9: baseline passed (pilot targets failures).
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |

Saved resumable outputs to: /kaggle/working/top2_counterfactual_pilot


In [1]:
"""Semantic-lookahead pilot for the exhaustive HumanEval/26 experiment.

Paste this complete file into a NEW Kaggle cell after running:
  1. the dependency-installation cell; and
  2. the large definitions cell from notebook0cef4050e3.ipynb.

Also add ``exhaustive_branch_humaneval_26_qwen25_7b.zip`` as a Kaggle
input (or leave its extracted output under /kaggle/working).

The experiment does not use tests to select positions. At every previously
evaluated position it forces the exact saved alternative token, rolls out only
12 tokens, and ranks the branch using pre-registered, label-free trajectory
features. The saved pass/fail outcome is used only after ranking for evaluation.
"""

import ast
import csv
import gc
import json
import keyword
import math
import re
import shutil
import time
import zipfile
from pathlib import Path
from typing import Any, Callable

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


LOOKAHEAD_TASK_ID = "HumanEval/26"
LOOKAHEAD_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
SOURCE_ARCHIVE_NAME = "exhaustive_branch_humaneval_26_qwen25_7b.zip"
LOOKAHEAD_TOKENS = 12
OUTPUT_DIR = Path("/kaggle/working/semantic_lookahead_pilot_humaneval_26")
EXTRACT_DIR = Path("/kaggle/working/semantic_lookahead_source")
OUTPUT_ZIP = Path(
    "/kaggle/working/semantic_lookahead_pilot_humaneval_26.zip"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def find_source_artifacts() -> Path:
    """Find the exhaustive pilot or safely extract its ZIP."""
    direct = Path(
        "/kaggle/working/exhaustive_branch_humaneval_26_qwen25_7b"
    )
    required = {"baseline.json", "exhaustive_branches.jsonl"}
    if direct.is_dir() and required.issubset(
        {path.name for path in direct.iterdir()}
    ):
        return direct

    candidates = []
    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if root.exists():
            candidates.extend(root.rglob(SOURCE_ARCHIVE_NAME))
    if not candidates:
        raise FileNotFoundError(
            f"Add {SOURCE_ARCHIVE_NAME} to Kaggle Input first."
        )

    archive_path = sorted(candidates, key=lambda path: len(str(path)))[0]
    print(f"Using source archive: {archive_path}")
    with zipfile.ZipFile(archive_path) as archive:
        names = {Path(name).name for name in archive.namelist()}
        if not required.issubset(names):
            raise RuntimeError(
                f"Source ZIP is missing: {sorted(required - names)}"
            )
        for info in archive.infolist():
            path = Path(info.filename)
            if path.is_absolute() or ".." in path.parts:
                raise RuntimeError(f"Unsafe ZIP path: {info.filename}")
        archive.extractall(EXTRACT_DIR)

    for candidate in (EXTRACT_DIR, *EXTRACT_DIR.iterdir()):
        if candidate.is_dir() and required.issubset(
            {path.name for path in candidate.iterdir()}
        ):
            return candidate
    raise RuntimeError("Could not locate extracted source artifacts.")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()


def token_class(text: str) -> str:
    """Coarse class for a code token; independent of test outcomes."""
    if not text:
        return "empty"
    if text.isspace() or "\n" in text:
        return "whitespace"
    stripped = text.strip()
    if keyword.iskeyword(stripped):
        return "keyword"
    if stripped.isidentifier():
        return "identifier"
    if stripped in {
        "+", "-", "*", "/", "//", "%", "**", "=", "==", "!=", "<", ">",
        "<=", ">=", ":=", "&", "|", "^", "~", "<<", ">>", "->",
    }:
        return "operator"
    try:
        ast.literal_eval(stripped)
        return "literal"
    except Exception:
        return "other"


def levenshtein_distance(left: list[int], right: list[int]) -> int:
    """Memory-efficient edit distance."""
    if len(left) < len(right):
        left, right = right, left
    previous = list(range(len(right) + 1))
    for i, left_value in enumerate(left, start=1):
        current = [i]
        for j, right_value in enumerate(right, start=1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (left_value != right_value),
                )
            )
        previous = current
    return previous[-1]


def aligned_mismatch(left: list[int], right: list[int], start: int = 0) -> float:
    length = max(len(left), len(right))
    if length <= start:
        return 0.0
    mismatches = 0
    for index in range(start, length):
        left_value = left[index] if index < len(left) else None
        right_value = right[index] if index < len(right) else None
        mismatches += left_value != right_value
    return mismatches / (length - start)


def weighted_token_divergence(
    baseline_texts: list[str],
    branch_texts: list[str],
) -> float:
    """Discount formatting and simple renames, emphasize code decisions."""
    length = max(len(baseline_texts), len(branch_texts))
    if length == 0:
        return 0.0
    total = 0.0
    for index in range(length):
        left = baseline_texts[index] if index < len(baseline_texts) else ""
        right = branch_texts[index] if index < len(branch_texts) else ""
        if left == right:
            continue
        left_class = token_class(left)
        right_class = token_class(right)
        classes = {left_class, right_class}
        if classes == {"whitespace"}:
            weight = 0.0
        elif classes == {"identifier"}:
            weight = 0.20
        elif "whitespace" in classes:
            weight = 0.25
        elif classes & {"keyword", "operator", "literal"}:
            weight = 1.0
        else:
            weight = 0.50
        total += weight
    return total / length


ANCHOR_OPERATOR_PATTERN = re.compile(
    r"==|!=|<=|>=|:=|//|\*\*|->|[+\-*/%<>=&|^~]"
)
ANCHOR_CALL_PATTERN = re.compile(
    r"\b([A-Za-z_]\w*)\s*\("
)
ANCHOR_METHOD_PATTERN = re.compile(
    r"\.\s*([A-Za-z_]\w*)\b"
)
ANCHOR_NUMBER_PATTERN = re.compile(
    r"(?<![\w.])(?:0[xob][0-9A-Fa-f]+|\d+(?:\.\d+)?)(?![\w.])"
)
ANCHOR_WORD_PATTERN = re.compile(r"\b[A-Za-z_]\w*\b")


def code_anchors(text: str) -> list[str]:
    """Extract intent-bearing lexical anchors while ignoring plain names."""
    anchors = []
    for word in ANCHOR_WORD_PATTERN.findall(text):
        if keyword.iskeyword(word):
            anchors.append(f"keyword:{word}")
    anchors.extend(
        f"operator:{value}" for value in ANCHOR_OPERATOR_PATTERN.findall(text)
    )
    anchors.extend(
        f"call:{value}" for value in ANCHOR_CALL_PATTERN.findall(text)
        if not keyword.iskeyword(value)
    )
    anchors.extend(
        f"method:{value}" for value in ANCHOR_METHOD_PATTERN.findall(text)
    )
    anchors.extend(
        f"number:{value}" for value in ANCHOR_NUMBER_PATTERN.findall(text)
    )
    return anchors


def anchor_features(baseline_text: str, branch_text: str) -> dict[str, float]:
    baseline = set(code_anchors(baseline_text))
    branch = set(code_anchors(branch_text))
    union = baseline | branch
    distance = 1.0 - len(baseline & branch) / len(union) if union else 0.0

    baseline_lexemes = max(len(ANCHOR_WORD_PATTERN.findall(baseline_text)), 1)
    branch_lexemes = max(len(ANCHOR_WORD_PATTERN.findall(branch_text)), 1)
    baseline_density = len(code_anchors(baseline_text)) / baseline_lexemes
    branch_density = len(code_anchors(branch_text)) / branch_lexemes

    # A prose/docstring detour can be very different but has low code density.
    density_retention = min(
        (branch_density + 0.05) / (baseline_density + 0.05),
        1.0,
    )
    return {
        "anchor_distance": distance,
        "baseline_anchor_density": baseline_density,
        "branch_anchor_density": branch_density,
        "anchor_score": distance * density_retention,
    }


@torch.inference_mode()
def generate_short_forced_rollout(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    forced_token_id: int,
    rollout_tokens: int,
) -> dict[str, Any]:
    """Measure branch probabilities and generate a short greedy continuation."""
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    if prefix_ids:
        prefix_tensor = torch.tensor(
            [prefix_ids], dtype=torch.long, device=input_device
        )
        context_ids = torch.cat([prompt_ids, prefix_tensor], dim=1)
    else:
        context_ids = prompt_ids

    started = time.perf_counter()
    outputs = model(context_ids, use_cache=True)
    cache = outputs.past_key_values
    logits = outputs.logits[0, -1, :].float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    top_probabilities, top_ids = torch.topk(probabilities, k=2)
    entropy = float(-(probabilities * log_probabilities).sum().item())
    forced_probability = float(probabilities[forced_token_id].item())
    forced_log_probability = float(log_probabilities[forced_token_id].item())
    forced_rank = int((logits > logits[forced_token_id]).sum().item()) + 1

    generated = [int(forced_token_id)]
    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None or forced_token_id != eos_token_id:
        next_input = torch.tensor(
            [[forced_token_id]], dtype=torch.long, device=input_device
        )
        outputs = model(next_input, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[0, -1, :]

        for _ in range(max(rollout_tokens - 1, 0)):
            next_id = int(torch.argmax(next_logits).item())
            generated.append(next_id)
            if eos_token_id is not None and next_id == eos_token_id:
                break
            next_input = torch.tensor(
                [[next_id]], dtype=torch.long, device=input_device
            )
            outputs = model(
                next_input,
                past_key_values=cache,
                use_cache=True,
            )
            cache = outputs.past_key_values
            next_logits = outputs.logits[0, -1, :]

    result = {
        "branch_token_ids": generated,
        "branch_token_texts": [
            tokenizer.decode([token_id], skip_special_tokens=False)
            for token_id in generated
        ],
        "same_run_top1_id": int(top_ids[0].item()),
        "same_run_top2_id": int(top_ids[1].item()),
        "same_run_p1": float(top_probabilities[0].item()),
        "same_run_p2": float(top_probabilities[1].item()),
        "same_run_probability_margin": float(
            top_probabilities[0].item() - top_probabilities[1].item()
        ),
        "same_run_entropy": entropy,
        "forced_probability": forced_probability,
        "forced_log_probability": forced_log_probability,
        "forced_rank": forced_rank,
        "latency_s": time.perf_counter() - started,
    }
    del outputs, logits, log_probabilities, probabilities
    return result


def ordinal_percentile_scores(
    rows: list[dict[str, Any]],
    field: str,
    descending: bool = True,
) -> dict[int, float]:
    """Map values to [0,1]; 1 is selected first. Ties keep token position."""
    ordered = sorted(
        rows,
        key=lambda row: (
            -float(row[field]) if descending else float(row[field]),
            int(row["position"]),
        ),
    )
    denominator = max(len(ordered) - 1, 1)
    return {
        int(row["position"]): 1.0 - index / denominator
        for index, row in enumerate(ordered)
    }


def ranking_metrics(
    rows: list[dict[str, Any]],
    field: str,
    descending: bool,
) -> dict[str, Any]:
    ranked = sorted(
        rows,
        key=lambda row: float(row[field]),
        reverse=descending,
    )
    labels = [int(row["top2_passed"]) for row in ranked]
    positives = sum(labels)
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        raise RuntimeError("Ranking evaluation needs both outcome classes.")

    average_precision = sum(
        sum(labels[:index + 1]) / (index + 1)
        for index, label in enumerate(labels)
        if label
    ) / positives
    positive_values = [
        float(row[field]) for row in rows if row["top2_passed"]
    ]
    negative_values = [
        float(row[field]) for row in rows if not row["top2_passed"]
    ]
    wins = 0.0
    for positive in positive_values:
        for negative in negative_values:
            if positive == negative:
                wins += 0.5
            elif (positive > negative) == descending:
                wins += 1.0

    rank_by_position = {
        int(row["position"]): rank
        for rank, row in enumerate(ranked, start=1)
    }
    return {
        "auroc": wins / (positives * negatives),
        "average_precision": average_precision,
        "positive_ranks": {
            str(row["position"]): rank_by_position[int(row["position"])]
            for row in rows
            if row["top2_passed"]
        },
        "top_k": {
            str(k): {
                "positions": [
                    int(row["position"]) for row in ranked[:k]
                ],
                "recoveries": sum(labels[:k]),
                "recall": sum(labels[:k]) / positives,
                "precision": sum(labels[:k]) / k,
            }
            for k in (1, 3, 5, 10, 15)
        },
    }


def make_plot(rows: list[dict[str, Any]], output_path: Path) -> bool:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return False

    methods = [
        ("entropy", "same_run_entropy", True),
        ("margin", "same_run_probability_margin", False),
        ("edit", "normalized_edit_distance", True),
        ("semantic", "semantic_lookahead_score", True),
    ]
    cutoffs = [1, 3, 5, 10, 15]
    values = []
    for label, field, descending in methods:
        ranked = sorted(
            rows,
            key=lambda row: float(row[field]),
            reverse=descending,
        )
        values.append(
            [
                sum(int(row["top2_passed"]) for row in ranked[:k])
                for k in cutoffs
            ]
        )

    x = np.arange(len(cutoffs))
    width = 0.19
    plt.figure(figsize=(9, 5.2))
    for index, ((label, _, _), counts) in enumerate(zip(methods, values)):
        plt.bar(x + (index - 1.5) * width, counts, width, label=label)
    plt.xticks(x, [f"top-{k}" for k in cutoffs])
    plt.yticks([0, 1, 2, 3])
    plt.ylabel("Recoveries found (out of 3)")
    plt.title("HumanEval/26: selection under a branch budget")
    plt.legend()
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()
    return True


def write_csv(rows: list[dict[str, Any]], path: Path) -> None:
    fields = [
        "position",
        "top1_token",
        "top2_token",
        "top2_passed",
        "same_run_top1_id",
        "same_run_top2_id",
        "forced_rank",
        "same_run_p1",
        "same_run_p2",
        "same_run_probability_margin",
        "same_run_entropy",
        "forced_probability",
        "normalized_edit_distance",
        "aligned_mismatch",
        "persistence_after_forced",
        "weighted_semantic_divergence",
        "anchor_distance",
        "baseline_anchor_density",
        "branch_anchor_density",
        "anchor_score",
        "semantic_lookahead_score",
        "saved_pair_matches_same_run",
        "short_rollout_matches_saved_branch",
        "baseline_snippet",
        "branch_snippet",
    ]
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in sorted(rows, key=lambda item: item["position"]):
            writer.writerow({field: row.get(field) for field in fields})


def run_semantic_lookahead_pilot(overwrite: bool = False) -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU accelerator.")

    source_dir = find_source_artifacts()
    baseline = json.loads(
        (source_dir / "baseline.json").read_text(encoding="utf-8")
    )
    branches = sorted(
        read_jsonl(source_dir / "exhaustive_branches.jsonl"),
        key=lambda row: int(row["position"]),
    )
    if len(branches) != 49 or len({row["position"] for row in branches}) != 49:
        raise RuntimeError("Expected the 49 unique exhaustive interventions.")
    if baseline["passed"]:
        raise RuntimeError("This pilot expects the saved failing baseline.")

    tasks = load_tasks(LOOKAHEAD_TASK_ID, num_tasks=1)
    if len(tasks) != 1 or tasks[0].task_id != LOOKAHEAD_TASK_ID:
        raise RuntimeError(f"Could not load exactly {LOOKAHEAD_TASK_ID}.")

    rollout_path = OUTPUT_DIR / "short_rollouts.jsonl"
    if overwrite and rollout_path.exists():
        rollout_path.unlink()
    completed = {
        int(row["position"]): row
        for row in read_jsonl(rollout_path)
    } if rollout_path.exists() else {}

    gc.collect()
    torch.cuda.empty_cache()
    torch.manual_seed(42)
    print(f"Loading {LOOKAHEAD_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(LOOKAHEAD_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    prompt = build_prompt(tasks[0], tokenizer)
    model = AutoModelForCausalLM.from_pretrained(
        LOOKAHEAD_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    baseline_ids = [int(value) for value in baseline["token_ids"]]
    branch_by_position = {
        int(row["position"]): row for row in branches
    }
    for index, position in enumerate(sorted(branch_by_position), start=1):
        if position in completed:
            print(f"[{index:2d}/49] t={position:2d} cached")
            continue
        source = branch_by_position[position]
        forced_id = int(source["top2_token_id"])
        rollout_length = min(
            LOOKAHEAD_TOKENS,
            max(len(baseline_ids) - position, 1),
        )
        generated = generate_short_forced_rollout(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            prefix_ids=baseline_ids[:position],
            forced_token_id=forced_id,
            rollout_tokens=rollout_length,
        )

        baseline_window = baseline_ids[position:position + rollout_length]
        branch_window = generated["branch_token_ids"]
        baseline_texts = [
            tokenizer.decode([token_id], skip_special_tokens=False)
            for token_id in baseline_window
        ]
        baseline_snippet = tokenizer.decode(
            baseline_window, skip_special_tokens=True
        )
        branch_snippet = tokenizer.decode(
            branch_window, skip_special_tokens=True
        )
        anchors = anchor_features(baseline_snippet, branch_snippet)
        saved_pair = {
            int(source["top1_token_id"]),
            int(source["top2_token_id"]),
        }
        same_run_pair = {
            generated["same_run_top1_id"],
            generated["same_run_top2_id"],
        }

        candidate_prefix = tokenizer.decode(
            baseline_ids[:position] + branch_window,
            skip_special_tokens=True,
        )
        saved_code = str(source.get("top2_code", ""))
        record = {
            "task_id": LOOKAHEAD_TASK_ID,
            "model": LOOKAHEAD_MODEL,
            "lookahead_tokens": rollout_length,
            "position": position,
            "top1_token": source["top1_token"],
            "top2_token": source["top2_token"],
            "top1_token_id": int(source["top1_token_id"]),
            "top2_token_id": forced_id,
            "top2_passed": bool(source["top2_passed"]),
            "baseline_token_ids": baseline_window,
            "baseline_token_texts": baseline_texts,
            "baseline_snippet": baseline_snippet,
            "branch_snippet": branch_snippet,
            "normalized_edit_distance": levenshtein_distance(
                baseline_window, branch_window
            ) / max(len(baseline_window), len(branch_window), 1),
            "aligned_mismatch": aligned_mismatch(
                baseline_window, branch_window
            ),
            "persistence_after_forced": aligned_mismatch(
                baseline_window, branch_window, start=1
            ),
            "weighted_semantic_divergence": weighted_token_divergence(
                baseline_texts,
                generated["branch_token_texts"],
            ),
            **anchors,
            **generated,
            "saved_pair_matches_same_run": saved_pair == same_run_pair,
            "short_rollout_matches_saved_branch": (
                saved_code.startswith(candidate_prefix)
            ),
        }
        append_jsonl(rollout_path, record)
        completed[position] = record
        print(
            f"[{index:2d}/49] t={position:2d} "
            f"edit={record['normalized_edit_distance']:.3f} "
            f"persist={record['persistence_after_forced']:.3f} "
            f"anchor={record['anchor_score']:.3f} "
            f"{'PASS' if record['top2_passed'] else 'FAIL'}"
        )

    rows = [completed[position] for position in sorted(completed)]
    if len(rows) != 49:
        raise RuntimeError(f"Only {len(rows)}/49 short rollouts completed.")

    # Fixed, label-free combination: equal mean of three within-task ranks.
    # Outcomes are not accessed until the ranking has been computed.
    component_fields = [
        "persistence_after_forced",
        "weighted_semantic_divergence",
        "anchor_score",
    ]
    component_scores = {
        field: ordinal_percentile_scores(rows, field, descending=True)
        for field in component_fields
    }
    for row in rows:
        position = int(row["position"])
        row["semantic_lookahead_score"] = float(
            np.mean([
                component_scores[field][position]
                for field in component_fields
            ])
        )

    selectors = {
        "entropy_high": ("same_run_entropy", True),
        "probability_margin_low": ("same_run_probability_margin", False),
        "edit_distance_high": ("normalized_edit_distance", True),
        "mismatch_high": ("aligned_mismatch", True),
        "persistence_high": ("persistence_after_forced", True),
        "weighted_semantic_high": (
            "weighted_semantic_divergence",
            True,
        ),
        "anchor_score_high": ("anchor_score", True),
        "semantic_lookahead_high": ("semantic_lookahead_score", True),
    }
    metrics = {
        name: ranking_metrics(rows, field, descending)
        for name, (field, descending) in selectors.items()
    }
    same_run_pair_mismatches = [
        int(row["position"])
        for row in rows
        if not row["saved_pair_matches_same_run"]
    ]
    rollout_mismatches = [
        int(row["position"])
        for row in rows
        if not row["short_rollout_matches_saved_branch"]
    ]

    report = {
        "status": "descriptive_single_problem_pilot",
        "task_id": LOOKAHEAD_TASK_ID,
        "model": LOOKAHEAD_MODEL,
        "evaluated_positions": len(rows),
        "recoveries": sum(int(row["top2_passed"]) for row in rows),
        "lookahead_tokens": LOOKAHEAD_TOKENS,
        "selection_uses_execution_tests": False,
        "combination_rule": {
            "type": "equal_mean_of_within_problem_ordinal_percentiles",
            "components": component_fields,
            "weights": [1 / 3, 1 / 3, 1 / 3],
            "weights_fitted_on_labels": False,
        },
        "metrics": metrics,
        "saved_pair_mismatch_positions": same_run_pair_mismatches,
        "saved_branch_rollout_mismatch_positions": rollout_mismatches,
        "decision_rule_for_next_stage": {
            "continue_if": (
                "semantic_lookahead finds >=1 recovery in top-5 AND "
                "beats entropy top-5; otherwise redesign the signal"
            ),
            "warning": (
                "This rule only checks plausibility on one problem and is "
                "not evidence of generalization."
            ),
        },
    }
    write_csv(rows, OUTPUT_DIR / "semantic_lookahead_positions.csv")
    (OUTPUT_DIR / "semantic_lookahead_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    make_plot(rows, OUTPUT_DIR / "semantic_lookahead_rankings.png")

    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()
    archive_base = str(OUTPUT_ZIP.with_suffix(""))
    shutil.make_archive(archive_base, "zip", root_dir=OUTPUT_DIR)

    print("\n=== SEMANTIC LOOKAHEAD REPORT ===")
    compact = {
        name: {
            "AUROC": round(values["auroc"], 3),
            "AP": round(values["average_precision"], 3),
            "positive_ranks": values["positive_ranks"],
            "top5_recoveries": values["top_k"]["5"]["recoveries"],
        }
        for name, values in metrics.items()
    }
    print(json.dumps(compact, indent=2, ensure_ascii=False))
    print("\nDownload this ZIP and send it back:")
    print(OUTPUT_ZIP)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


run_semantic_lookahead_pilot(overwrite=False)

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading Qwen/Qwen2.5-Coder-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[ 1/49] t= 2 cached
[ 2/49] t= 3 cached
[ 3/49] t= 4 cached
[ 4/49] t= 5 cached
[ 5/49] t= 6 cached
[ 6/49] t= 7 cached
[ 7/49] t= 8 cached
[ 8/49] t= 9 cached
[ 9/49] t=10 cached
[10/49] t=11 cached
[11/49] t=12 cached
[12/49] t=13 cached
[13/49] t=14 cached
[14/49] t=15 cached
[15/49] t=16 cached
[16/49] t=17 cached
[17/49] t=18 cached
[18/49] t=19 cached
[19/49] t=20 cached
[20/49] t=21 cached
[21/49] t=22 cached
[22/49] t=23 cached
[23/49] t=24 cached
[24/49] t=25 cached
[25/49] t=26 cached
[26/49] t=27 cached
[27/49] t=28 cached
[28/49] t=29 cached
[29/49] t=30 cached
[30/49] t=31 cached
[31/49] t=32 cached
[32/49] t=33 cached
[33/49] t=34 cached
[34/49] t=35 cached
[35/49] t=36 cached
[36/49] t=37 cached
[37/49] t=38 cached
[38/49] t=39 cached
[39/49] t=40 cached
[40/49] t=41 cached
[41/49] t=42 cached
[42/49] t=43 cached
[43/49] t=44 cached
[44/49] t=45 cached
[45/49] t=46 cached
[46/49] t=47 cached
[47/49] t=48 cached
[48/49] t=49 cached
[49/49] t=50 cached

=== SEMANTIC LOOKAH